In [ ]:
def setup_dataset():
    data_config = {
        "path": "./worm_dataset",
        "train": "train/images",
        "val": "val/images",
        "test": "val/images",
        "names": ["worm"],
        "nc": 1
    }
    with open("dataset.yaml", "w") as f:
        yaml.dump(data_config, f)
    print("✅ dataset.yaml created")

def draw_detections(result):
    img = result.orig_img.copy()
    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])
        label = result.names[int(box.cls[0])]
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(img, f"{label} {conf:.2f}", (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
    return img

In [2]:
from onnxslim.core import freeze
from torch.optim.adamw import AdamW
from ultralytics import YOLO
from win32comext.shell.demos.servers.folder_view import tasks


def main():
    # Ensure dataset.yaml exists and is correct
    setup_dataset()

    # Load a YOLOv8 model (nano is fastest, switch to 'yolov8s' or 'yolov8m' for better accuracy)
    model = YOLO("yolo12m.pt")  # Pretrained base model

    ls=range(64)
    # Train the model
    model.train(task='detect', data="dataset.yaml",freeze=ls, device=3, patience=75,epochs=1500, imgsz=640, batch=8,single_cls=True,overlap_mask=False,cache="disk", box=12.0,plots=True,workers=32)

    # Save final weights
    model.export(format="onnx")  # Optional export
    print("Training completed!")

if __name__ == "__main__":
    main()
